In [ ]:
# =========================================================
# ENTERPRISE RAG CACHE SYSTEM
# =========================================================
#
# FEATURES:
# ✅ FAQ Cache (Top 100 Questions)
# ✅ Redis Dynamic Cache
# ✅ Query Frequency Tracking
# ✅ LLM Fallback
# ✅ Automatic Popular Query Caching
# ✅ TTL Expiry
#
# =========================================================

import redis
import hashlib
import json


# =========================================================
# REDIS CONNECTION
# =========================================================

r = redis.Redis(
    host='localhost',
    port=6379,
    decode_responses=True
)


# =========================================================
# LOAD TOP FAQ QUESTIONS
# (You can keep 100+ questions here)
# =========================================================

FAQ_CACHE = {

    "what is leave policy?":
        "Employees get 20 paid leaves every year.",

    "what is wfh policy?":
        "Employees can work from home 2 days per week.",

    "office timings":
        "Office timings are 9 AM to 6 PM.",

    "what is attendance policy?":
        "Employees must maintain 90% attendance.",

    "what is payroll date?":
        "Salary is credited on the last working day.",

    "how many casual leaves allowed?":
        "Employees get 10 casual leaves annually.",

    "what is laptop policy?":
        "Company-issued laptops must be returned during exit.",

    "what is insurance policy?":
        "Health insurance is provided to all full-time employees.",

    "how many sick leaves allowed?":
        "Employees get 8 sick leaves annually.",

    "what is resignation notice period?":
        "Notice period is 60 days."
}


# =========================================================
# PRELOAD FAQ CACHE INTO REDIS
# (Runs once during startup)
# =========================================================

def preload_faq_cache():

    for question, answer in FAQ_CACHE.items():

        redis_key = f"faq:{question}"

        r.set(redis_key, answer)

    print("FAQ CACHE LOADED INTO REDIS")


# =========================================================
# MOCK LLM FUNCTION
# Replace with OpenAI / Groq / Azure OpenAI etc.
# =========================================================

def llm_response(query):

    # Simulate AI Response
    return f"LLM Generated Answer For: {query}"


# =========================================================
# GENERATE HASH KEY
# =========================================================

def generate_hash(text):

    return hashlib.md5(text.encode()).hexdigest()


# =========================================================
# MAIN QUERY FUNCTION
# =========================================================

def get_answer(query):

    # Normalize query
    normalized_query = query.lower().strip()


    # =====================================================
    # STEP 1 → CHECK FAQ CACHE
    # =====================================================

    faq_key = f"faq:{normalized_query}"

    faq_answer = r.get(faq_key)

    if faq_answer:

        print("\n✅ FAQ CACHE HIT")

        return faq_answer


    # =====================================================
    # STEP 2 → CHECK DYNAMIC REDIS CACHE
    # =====================================================

    response_key = f"response:{generate_hash(normalized_query)}"

    cached_response = r.get(response_key)

    if cached_response:

        print("\n✅ REDIS DYNAMIC CACHE HIT")

        return cached_response


    # =====================================================
    # STEP 3 → CACHE MISS → CALL LLM
    # =====================================================

    print("\n❌ CACHE MISS → CALLING LLM")

    answer = llm_response(query)


    # =====================================================
    # STEP 4 → TRACK QUERY FREQUENCY
    # =====================================================

    count_key = f"count:{generate_hash(normalized_query)}"

    query_count = r.incr(count_key)

    print(f"📌 Query Count = {query_count}")


    # =====================================================
    # STEP 5 → STORE ONLY FREQUENT QUESTIONS
    #
    # Example:
    # Cache only after query repeated 3 times
    # =====================================================

    if query_count >= 3:

        print("🔥 STORING RESPONSE IN REDIS CACHE")

        # Store for 1 hour
        r.setex(
            response_key,
            3600,
            answer
        )


    # =====================================================
    # RETURN FINAL ANSWER
    # =====================================================

    return answer


# =========================================================
# APPLICATION STARTUP
# =========================================================

preload_faq_cache()


# =========================================================
# MAIN LOOP
# =========================================================

while True:

    print("\n==============================")

    query = input("Ask Question (type 'exit' to quit): ")


    if query.lower() == "exit":

        print("\nApplication Closed")

        break


    response = get_answer(query)

    print("\n💬 FINAL RESPONSE:")
    print(response)